In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("../data/processed/products_train_clean.csv")
test = pd.read_csv("../data/processed/products_test_clean.csv")
for d in (train, test):
    d["log_price"] = np.log1p(d["price"])

top = train["main_category"].value_counts().head(5).index

def make_X(d):
    X = pd.DataFrame({"desc_length": d["desc_length"]})
    for c in top:
        X[c] = (d["main_category"] == c).astype(float)
    return X

Xtr, Xte = make_X(train).to_numpy(), make_X(test).to_numpy()
ytr, yte = train["log_price"].to_numpy(), test["log_price"].to_numpy()
Xtr.shape, Xte.shape

((23640, 6), (6547, 6))

In [2]:
Xtr1 = np.c_[np.ones(len(Xtr)), Xtr]   # add a column of 1s for the intercept
Xte1 = np.c_[np.ones(len(Xte)), Xte]

w = np.linalg.solve(Xtr1.T @ Xtr1, Xtr1.T @ ytr)
print(w)

[2.92051480e+00 9.45443626e-04 5.01302742e-02 7.09334548e-01
 1.11626597e+00 2.61817485e-01 1.14350267e-01]


In [3]:
def r2(y, p):
    return 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()

pred = Xte1 @ w
print("From-scratch test R²:", round(r2(yte, pred), 4))

From-scratch test R²: 0.1141


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

lr = LinearRegression().fit(Xtr, ytr)
print("sklearn intercept + coefs:", lr.intercept_, lr.coef_)
print("sklearn test R²:", round(r2_score(yte, lr.predict(Xte)), 4))
print("my weights:", w)

sklearn intercept + coefs: 2.9205147955679913 [9.45443626e-04 5.01302742e-02 7.09334548e-01 1.11626597e+00
 2.61817485e-01 1.14350267e-01]
sklearn test R²: 0.1141
my weights: [2.92051480e+00 9.45443626e-04 5.01302742e-02 7.09334548e-01
 1.11626597e+00 2.61817485e-01 1.14350267e-01]
